# Ozon E-CUP 2026 — E5 V3
## Fashion specialist + variant signals + hard mining + hybrid routing

Это **следующий эксперимент после E5-small V2**, который дал наш лучший leaderboard:

- Public LB Mean PR-AUC: **0.4838757641**
- V2 backbone: `intfloat/multilingual-e5-small`
- V2 checkpoint: step `30000`
- V2 full LLM group holdout Macro PR-AUC: примерно **0.78648**
- V2 `MAX_LEN=192`

### Почему V3 не обучается с нуля

V2 уже хорошо работает на большинстве категорий. Основной резерв сосредоточен в fashion:
- `Обувь`
- `Одежда`
- `Галантерея и аксессуары`
- `Ювелирные изделия`

Поэтому V3 строится как **второй эксперт**:
- base E5 V2 остаётся для категорий, где он сильнее;
- fashion-specialist стартует с тех же весов V2 и дообучается только на fashion LLM-парах;
- итоговый `routing.json` выбирает лучшего эксперта отдельно для каждой fashion-категории.

### Что нового в V3

1. Исправлены точные имена fashion-категорий.
2. Используется только LLM distribution; manual stage-B нет.
3. Fashion train pool сбалансирован по категориям.
4. Hard-example mining от текущего V2.
5. В текст добавляются **явные pair-level variant signals**:
   - размер,
   - цвет,
   - артикул / SKU / OEM,
   - модель,
   - пол,
   - материал.
6. Soft BCE остаётся confidence-aware.
7. Частая validation.
8. Полноценный resume checkpoint содержит model + optimizer + scheduler + scaler + RNG states.
9. Best checkpoint выбирается по **hybrid Macro PR-AUC**, то есть ровно по логике будущего submission.
10. Финальный export содержит base model + specialist + tokenizer + routing config.

Цель — не просто пробить `0.50`, а продолжать сокращать разрыв с лидерами leaderboard (~`0.55` на момент проектирования V3).

## Перед запуском

В Kaggle вручную:
- Accelerator → GPU
- Internet → On

Нужные inputs:

```text
/kaggle/input/datasets/mihailivanovvvv/first-model-ozon-ecup-torch-checpoint
/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items
```

Notebook **не содержит Kaggle-specific metadata** и не меняет Settings сам.

In [1]:
import os
import gc
import re
import json
import math
import time
import random
import shutil
import warnings
import zipfile
from pathlib import Path
from collections import defaultdict

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import average_precision_score
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
DATA_ROOT = "/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items"
CKPT_DATASET_ROOT = Path(
    "/kaggle/input/datasets/mihailivanovvvv/"
    "first-model-ozon-ecup-torch-checkpoint"
)

ITEMS_PATH = f"{DATA_ROOT}/items.parquet"
ITEMS_HUMAN_PATH = f"{DATA_ROOT}/items_human.parquet"
MATCHES_PATH = f"{DATA_ROOT}/matches.parquet"
MATCHES_LLM_PATH = f"{DATA_ROOT}/matches_llm.parquet"

# ------------------------------------------------------------
# V2 reference
# ------------------------------------------------------------
MODEL_NAME = "intfloat/multilingual-e5-small"
EXPECTED_V2_STEP = 30000
EXPECTED_V2_FAST_MACRO = 0.7863788901130114
EXPECTED_V2_FULL_MACRO = 0.786482334
V2_PUBLIC_LB = 0.4838757641

MAX_LEN = 192
MAX_ATTR_CHARS = 460

# ------------------------------------------------------------
# Exact fashion categories — V2 bug fixed here
# ------------------------------------------------------------
FASHION_CATEGORIES = {
    "Обувь",
    "Одежда",
    "Галантерея и аксессуары",
    "Ювелирные изделия",
}

# ------------------------------------------------------------
# V3 mining / training
# ------------------------------------------------------------
SEED = 42

# Balanced candidate pool: up to 200k per category.
CANDIDATE_PER_CATEGORY = 200_000

# Selected train set after mining: up to 150k per category.
TRAIN_PER_CATEGORY = 150_000

HARD_FRAC = 0.50
RANDOM_FRAC = 0.30
STABLE_FRAC = 0.20

SOURCE_WEIGHTS = {
    "hard": 1.25,
    "random": 1.00,
    "stable": 0.80,
}

# Fine-tune from already trained V2.
EPOCHS = 2
BACKBONE_LR = 8e-6
HEAD_LR = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.04
MAX_GRAD_NORM = 1.0

RANDOM_SWAP_PROB = 0.50
CONFIDENCE_FLOOR = 0.75
CONFIDENCE_POWER = 1.0

# Routing safety margin: specialist must beat base by at least this much
# on fast validation to be selected for a category.
GATE_MARGIN = 0.002

# ------------------------------------------------------------
# Same LLM split as V2
# ------------------------------------------------------------
LLM_VAL_FRAC = 0.03
LLM_VAL_SEED = 13
FAST_VAL_PER_CATEGORY = 3000
FAST_VAL_SAMPLE_SEED = 0

# Frequent eval + resume.
EVAL_EVERY_OPT_STEPS = 500
RESUME_EVERY_OPT_STEPS = 500

BEST_PATH = "/kaggle/working/e5_v3_best_specialist.pt"
RESUME_PATH = "/kaggle/working/e5_v3_resume.pt"
EXPORT_ROOT = "/kaggle/working/e5_v3_hybrid_export"

DEBUG = False
if DEBUG:
    CANDIDATE_PER_CATEGORY = 5_000
    TRAIN_PER_CATEGORY = 3_000
    EPOCHS = 1
    EVAL_EVERY_OPT_STEPS = 50
    RESUME_EVERY_OPT_STEPS = 50

# ------------------------------------------------------------
# Reproducibility / GPU
# ------------------------------------------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "Включи GPU Accelerator в Kaggle."
device = torch.device("cuda")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

# T4 from previous run reports ~14.6 GB, so it falls into the safe branch.
if gpu_mem_gb >= 35:
    MICRO_BATCH = 96
    GRAD_ACCUM = 2
    PRED_BATCH = 192
elif gpu_mem_gb >= 14:
    MICRO_BATCH = 32
    GRAD_ACCUM = 8
    PRED_BATCH = 64
else:
    MICRO_BATCH = 24
    GRAD_ACCUM = 8
    PRED_BATCH = 48

EFFECTIVE_BATCH = MICRO_BATCH * GRAD_ACCUM

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("GPU:", gpu_name)
print(f"GPU memory: {gpu_mem_gb:.1f} GB")
print("micro batch:", MICRO_BATCH)
print("grad accum:", GRAD_ACCUM)
print("effective batch:", EFFECTIVE_BATCH)
print("prediction batch:", PRED_BATCH)
print("debug:", DEBUG)

GPU: Tesla T4
GPU memory: 14.6 GB
micro batch: 32
grad accum: 8
effective batch: 256
prediction batch: 64
debug: False


## 1. Проверяем inputs и восстанавливаем V2 checkpoint

Kaggle распаковывает `torch.save` archive как dataset directory. Поэтому автоматически собираем его обратно в `/kaggle/working/e5_macro_v2_best_repacked.pt`.

In [2]:
for p in [ITEMS_PATH, ITEMS_HUMAN_PATH, MATCHES_PATH, MATCHES_LLM_PATH]:
    assert os.path.exists(p), f"Не найден data file: {p}"
    print(f"{os.path.basename(p):24s} {os.path.getsize(p)/1024**3:8.3f} GB")

assert CKPT_DATASET_ROOT.exists(), CKPT_DATASET_ROOT

data_pkl_files = list(CKPT_DATASET_ROOT.rglob("data.pkl"))
assert data_pkl_files, "Не найден data.pkl внутри checkpoint dataset."

extracted_root = data_pkl_files[0].parent
print("\nDetected extracted checkpoint root:")
print(extracted_root)

CKPT_PATH = Path("/kaggle/working/e5_macro_v2_best_repacked.pt")

prefix = "e5_macro_v2_best"
with zipfile.ZipFile(CKPT_PATH, "w", compression=zipfile.ZIP_STORED) as zf:
    for file_path in extracted_root.rglob("*"):
        if not file_path.is_file():
            continue
        relative = file_path.relative_to(extracted_root)
        zf.write(file_path, arcname=str(Path(prefix) / relative))

print("\nReconstructed checkpoint:", CKPT_PATH)
print(f"size: {os.path.getsize(CKPT_PATH)/1024**2:.1f} MB")

v2_ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

print("\ncheckpoint keys:", v2_ckpt.keys())
print("model_name:", v2_ckpt.get("model_name"))
print("opt_step:", v2_ckpt.get("opt_step"))
print("metric:", v2_ckpt.get("metric"))
print("max_len:", v2_ckpt.get("max_len"))
print("model tensors:", len(v2_ckpt["model"]))

assert v2_ckpt["model_name"] == MODEL_NAME
assert v2_ckpt["opt_step"] == EXPECTED_V2_STEP
assert int(v2_ckpt["max_len"]) == MAX_LEN
assert "classifier.weight" in v2_ckpt["model"]
assert "classifier.bias" in v2_ckpt["model"]

print("\nV2 checkpoint OK.")

items.parquet               3.822 GB
items_human.parquet         0.199 GB
matches.parquet             0.004 GB
matches_llm.parquet         0.098 GB

Detected extracted checkpoint root:
/kaggle/input/datasets/mihailivanovvvv/first-model-ozon-ecup-torch-checkpoint/e5_macro_v2_best

Reconstructed checkpoint: /kaggle/working/e5_macro_v2_best_repacked.pt
size: 448.9 MB

checkpoint keys: dict_keys(['model', 'metric', 'opt_step', 'model_name', 'max_len', 'category_weight'])
model_name: intfloat/multilingual-e5-small
opt_step: 30000
metric: 0.7863788901130114
max_len: 192
model tensors: 201

V2 checkpoint OK.


## 2. Тот же group split, что в V2

Случайный pair split не используем. Validation строится union-find по связанным товарам.

Важно: здесь сохраняем `LLM_VAL_SEED=13`, потому что именно на этом split обучался и валидировался наш V2, который дал LB `0.4838757641`.

In [3]:
def group_val_mask(df, val_frac, seed):
    parent = {}

    def find(x):
        p = parent.setdefault(x, x)
        while p != parent[p]:
            parent[p] = parent[parent[p]]
            p = parent[p]
        parent[x] = p
        return p

    for a, b in zip(df.id1.values, df.id2.values):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    comp = np.fromiter(
        (find(i) for i in df.id1.values),
        dtype=np.int64,
        count=len(df),
    )

    rng = np.random.RandomState(seed)
    uniq = np.unique(comp)
    val_set = set(uniq[rng.rand(len(uniq)) < val_frac].tolist())

    return np.fromiter(
        (c in val_set for c in comp),
        dtype=bool,
        count=len(df),
    )

t0 = time.time()

llm_all = pd.read_parquet(MATCHES_LLM_PATH)
print("LLM all:", len(llm_all))

llm_val_mask = group_val_mask(
    llm_all,
    LLM_VAL_FRAC,
    LLM_VAL_SEED,
)

llm_val_raw = llm_all[llm_val_mask].copy()

# Same confident subset used by V2 metric.
llm_val = llm_val_raw[
    (llm_val_raw.target <= 0.2) |
    (llm_val_raw.target >= 0.8)
].copy()
llm_val["target"] = (llm_val["target"] >= 0.5).astype(np.int8)
llm_val["_val_pos"] = np.arange(len(llm_val), dtype=np.int64)

print("LLM val raw:", len(llm_val_raw))
print("LLM val confident:", len(llm_val))
print(f"split time: {(time.time()-t0)/60:.2f} min")

LLM all: 11187780
LLM val raw: 237386
LLM val confident: 191555
split time: 0.75 min


## 3. Категории без полного text preprocessing

Сначала читаем из `items.parquet` только `id + category`.

Это позволяет:
- присвоить category full LLM holdout;
- выделить fashion train rows;
- только после этого строить тексты для реально нужных товаров.

In [4]:
t0 = time.time()

item_categories = pd.read_parquet(
    ITEMS_PATH,
    columns=["id", "category"],
)

print("items category rows:", len(item_categories))
print("categories:", sorted(item_categories["category"].dropna().unique()))

all_cat_map = item_categories.set_index("id")["category"]

llm_val["category"] = llm_val["id1"].map(all_cat_map)
assert llm_val["category"].notna().all()

fashion_items = item_categories[
    item_categories["category"].isin(FASHION_CATEGORIES)
][["id", "category"]].copy()

fashion_item_ids = pd.Index(fashion_items["id"].values)
fashion_cat_map = fashion_items.set_index("id")["category"]

# Fashion rows only from TRAIN side of group split.
train_fashion_mask = (~llm_val_mask) & llm_all["id1"].isin(fashion_item_ids)
fashion_train_all = llm_all.loc[
    train_fashion_mask,
    ["id1", "id2", "target"],
].copy()

fashion_train_all["category"] = fashion_train_all["id1"].map(fashion_cat_map)
assert fashion_train_all["category"].notna().all()

print("\nFashion train rows:", len(fashion_train_all))
display(
    fashion_train_all.groupby("category", observed=True)
    .size()
    .rename("pairs")
    .reset_index()
    .sort_values("pairs", ascending=False)
)

print(f"\ncategory mapping time: {(time.time()-t0)/60:.2f} min")

# Large structures no longer needed.
del fashion_items, fashion_item_ids, fashion_cat_map, all_cat_map, item_categories
del llm_val_raw
gc.collect()

items category rows: 13397761
categories: ['Автотовары', 'Аптека', 'Бытовая техника', 'Бытовая химия', 'Галантерея и аксессуары', 'Детские товары', 'Дом и сад', 'Канцелярские товары', 'Красота и гигиена', 'Мебель', 'Музыкальные инструменты', 'Обувь', 'Одежда', 'Продукты питания', 'Спорт и отдых', 'Строительство и ремонт', 'Товары для животных', 'Хобби и творчество', 'Электроника', 'Ювелирные изделия']

Fashion train rows: 2227727


,category,pairs
2,Одежда,584258
1,Обувь,582863
0,Галантерея и аксессуары,538595
3,Ювелирные изделия,522011



category mapping time: 0.13 min


160

## 4. Balanced fashion candidate pool

Hard mining будет выполняться не на всех fashion-парах, а на сбалансированном pool:

- максимум `200k` пар на категорию;
- максимум около `800k` пар всего.

Это делает эксперимент существенно быстрее полного повторного stage-A.

In [5]:
def balanced_sample_by_category(df, n_per_category, seed):
    parts = []
    for cat, g in df.groupby("category", observed=True):
        n = min(n_per_category, len(g))
        parts.append(g.sample(n=n, random_state=seed))
    return pd.concat(parts, ignore_index=True)

candidate_pool = balanced_sample_by_category(
    fashion_train_all,
    CANDIDATE_PER_CATEGORY,
    SEED,
)

candidate_pool["_candidate_pos"] = np.arange(len(candidate_pool), dtype=np.int64)

print("candidate pool:", len(candidate_pool))
display(
    candidate_pool.groupby("category", observed=True)
    .size()
    .rename("pairs")
    .reset_index()
)

del fashion_train_all
gc.collect()

candidate pool: 800000


,category,pairs
0,Галантерея и аксессуары,200000
1,Обувь,200000
2,Одежда,200000
3,Ювелирные изделия,200000


0

## 5. Загружаем только товары, нужные для candidate pool + full validation

Это экономит RAM относительно V2, где тексты строились для всех 13.4M товаров.

In [6]:
needed_ids = set(candidate_pool.id1.values.tolist())
needed_ids.update(candidate_pool.id2.values.tolist())
needed_ids.update(llm_val.id1.values.tolist())
needed_ids.update(llm_val.id2.values.tolist())

print("unique required items:", f"{len(needed_ids):,}")

unique required items: 1,354,301


## 6. Preprocessing V2 + variant extraction V3

Base text **не меняется**, чтобы V2 predictions воспроизводились.

Дополнительно для specialist извлекаем компактные variant fields:
- size
- color
- article/SKU/OEM
- model
- gender
- material

Variant signals добавляются только при specialist fine-tune/inference.

In [7]:
SPACE_RE = re.compile(r"\s+")
MULTIPLY_RE = re.compile(r"[×хХ]")

KEY_ORDER = [
    "бренд", "brand",
    "артикул", "партномер", "part number", "partnumber", "oem",
    "код", "sku", "модель", "model",
    "размер", "size", "рост", "обхват", "пол", "gender",
    "цвет", "color", "материал", "material", "сезон",
    "объем", "обьем", "volume", "вес", "weight",
    "длина", "ширина", "высота",
    "количество", "комплектация", "упаков",
    "тип", "type",
]

VARIANT_ALIASES = {
    "size": [
        "размер производителя",
        "размер обуви",
        "размер одежды",
        "размер",
        "size",
        "рост",
    ],
    "color": [
        "основной цвет",
        "цвет",
        "color",
        "расцветка",
    ],
    "article": [
        "артикул",
        "sku",
        "oem",
        "партномер",
        "part number",
        "partnumber",
        "код производителя",
    ],
    "model": [
        "модель",
        "model",
    ],
    "gender": [
        "пол",
        "gender",
    ],
    "material": [
        "материал верха",
        "материал",
        "material",
    ],
}

COMMON_COLORS = {
    "черный": "черный",
    "чёрный": "черный",
    "белый": "белый",
    "серый": "серый",
    "серебристый": "серебристый",
    "красный": "красный",
    "бордовый": "бордовый",
    "синий": "синий",
    "голубой": "голубой",
    "зеленый": "зеленый",
    "зелёный": "зеленый",
    "желтый": "желтый",
    "жёлтый": "желтый",
    "оранжевый": "оранжевый",
    "розовый": "розовый",
    "фиолетовый": "фиолетовый",
    "бежевый": "бежевый",
    "коричневый": "коричневый",
    "золотой": "золотой",
    "золотистый": "золотой",
    "разноцветный": "разноцветный",
}

def normalize_piece(x):
    if x is None:
        return ""
    s = str(x).lower().replace("ё", "е")
    s = MULTIPLY_RE.sub("x", s)
    s = s.replace(",", ".")
    s = SPACE_RE.sub(" ", s).strip()
    return s

def safe_attrs(attributes):
    if isinstance(attributes, dict):
        obj = attributes
    elif isinstance(attributes, str):
        try:
            obj = json.loads(attributes)
        except Exception:
            obj = {}
    else:
        obj = {}

    if not isinstance(obj, dict):
        return {}

    out = {}
    for k, v in obj.items():
        kk = normalize_piece(k)
        vv = normalize_piece(v)
        if kk and vv:
            out[kk] = vv
    return out

def build_text_v2(name, attributes, category, max_attr_chars=MAX_ATTR_CHARS):
    cat = normalize_piece(category)
    nm = normalize_piece(name)
    attrs = safe_attrs(attributes)

    picked = []
    used = set()

    for want in KEY_ORDER:
        for k, v in attrs.items():
            if k in used:
                continue
            if want in k:
                picked.append(f"{k}: {v}")
                used.add(k)

    rest = [f"{k}: {v}" for k, v in attrs.items() if k not in used]
    attr_text = " ; ".join(picked + rest)[:max_attr_chars]

    return f"категория: {cat} | название: {nm} | атрибуты: {attr_text}"

def pick_attr(attrs, aliases):
    for alias in aliases:
        for k, v in attrs.items():
            if alias in k:
                return v
    return ""

def canonical_compact(value):
    s = normalize_piece(value)
    if not s:
        return ""

    # Keep alphanumerics and basic separators, remove accidental formatting noise.
    s = re.sub(r"[^0-9a-zа-я.+/_\- ]+", " ", s)
    s = SPACE_RE.sub(" ", s).strip()

    return s[:80]

def canonical_code(value):
    s = canonical_compact(value)
    return re.sub(r"[^0-9a-zа-я]+", "", s)

def canonical_size(value):
    s = canonical_compact(value)
    if not s:
        return ""

    # Normalize common separators but do NOT translate S/M/L into numeric sizes.
    s = s.replace("–", "-").replace("—", "-")
    s = re.sub(r"\s*-\s*", "-", s)
    s = re.sub(r"\s*/\s*", "/", s)
    return s[:60]

def canonical_color(value):
    s = canonical_compact(value)
    if not s:
        return ""

    found = []
    for raw, canon in COMMON_COLORS.items():
        if raw.replace("ё", "е") in s:
            found.append(canon)

    if found:
        return "/".join(sorted(set(found)))

    # Unknown color names are still useful as normalized strings.
    return s[:60]

SIZE_NAME_RE = re.compile(
    r"(?:размер|р-р|size)\s*[:=\-]?\s*"
    r"([0-9]{2,3}(?:[./-][0-9]{1,3})?|"
    r"xxxs|xxs|xs|s|m|l|xl|xxl|xxxl)",
    flags=re.IGNORECASE,
)

def fallback_size_from_name(name):
    nm = normalize_piece(name)
    m = SIZE_NAME_RE.search(nm)
    return canonical_size(m.group(1)) if m else ""

def fallback_color_from_name(name):
    nm = normalize_piece(name)
    found = []
    for raw, canon in COMMON_COLORS.items():
        if raw.replace("ё", "е") in nm:
            found.append(canon)
    return "/".join(sorted(set(found)))

def extract_variant_tuple(name, attributes):
    attrs = safe_attrs(attributes)

    size = canonical_size(pick_attr(attrs, VARIANT_ALIASES["size"]))
    if not size:
        size = fallback_size_from_name(name)

    color = canonical_color(pick_attr(attrs, VARIANT_ALIASES["color"]))
    if not color:
        color = fallback_color_from_name(name)

    article = canonical_code(pick_attr(attrs, VARIANT_ALIASES["article"]))
    model = canonical_code(pick_attr(attrs, VARIANT_ALIASES["model"]))
    gender = canonical_compact(pick_attr(attrs, VARIANT_ALIASES["gender"]))
    material = canonical_compact(pick_attr(attrs, VARIANT_ALIASES["material"]))

    return (size, color, article, model, gender, material)

VARIANT_NAMES = ("размер", "цвет", "артикул", "модель", "пол", "материал")

def compare_variant_value(a, b):
    if not a or not b:
        return "нет данных"

    if a == b:
        return "совпадает"

    # Some multi-valued normalized fields may have overlap.
    sa = {x for x in re.split(r"[/|; ]+", a) if x}
    sb = {x for x in re.split(r"[/|; ]+", b) if x}

    if sa and sb and sa.intersection(sb):
        return "частично совпадает"

    return "различается"

def make_pair_signal(id1, id2):
    v1 = item_variant[id1]
    v2 = item_variant[id2]

    parts = []
    for name, a, b in zip(VARIANT_NAMES, v1, v2):
        status = compare_variant_value(a, b)

        if name in {"размер", "цвет", "артикул", "модель"} and a and b:
            parts.append(f"{name}: {a} vs {b} => {status}")
        else:
            parts.append(f"{name}: {status}")

    return " | сравнение вариантов: " + " ; ".join(parts)

In [8]:
t0 = time.time()

item_text = {}
item_variant = {}
item_category = {}

pf = pq.ParquetFile(ITEMS_PATH)

for batch_id, batch in enumerate(
    pf.iter_batches(
        columns=["id", "name", "attributes", "category"],
        batch_size=400_000,
    ),
    start=1,
):
    pdf = batch.to_pandas()
    sub = pdf[pdf["id"].isin(needed_ids)]

    for i, n, a, c in sub.itertuples(index=False, name=None):
        item_text[i] = build_text_v2(n, a, c)
        item_variant[i] = extract_variant_tuple(n, a)
        item_category[i] = c

    if batch_id % 5 == 0:
        print(
            f"batches={batch_id:3d} "
            f"found={len(item_text):,}/{len(needed_ids):,} "
            f"time={time.time()-t0:.0f}s",
            flush=True,
        )

    del pdf, sub, batch
    gc.collect()

missing = needed_ids - set(item_text)

print(f"\nloaded required items: {len(item_text):,}")
print("missing:", len(missing))
print(f"time: {(time.time()-t0)/60:.2f} min")

assert not missing

del needed_ids
gc.collect()

batches=  5 found=37,984/1,354,301 time=31s
batches= 10 found=375,555/1,354,301 time=111s
batches= 15 found=417,167/1,354,301 time=135s
batches= 20 found=733,839/1,354,301 time=225s
batches= 25 found=1,100,825/1,354,301 time=330s
batches= 30 found=1,148,049/1,354,301 time=362s

loaded required items: 1,354,301
missing: 0
time: 7.33 min


0

## 7. Restore base V2 model + tokenizer

Base model нужен для:
- sanity check,
- full validation predictions,
- hard-example mining.

Specialist позже создаётся из **тех же V2 weights**.

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=1)

base_model = AutoModelForSequenceClassification.from_config(config)
base_model.load_state_dict(v2_ckpt["model"], strict=True)
base_model = base_model.to(device)
base_model.eval()

n_params = sum(p.numel() for p in base_model.parameters())
print(f"parameters: {n_params/1e6:.1f}M")
print("base restored.")

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

parameters: 117.7M
base restored.


## 8. Inference helpers + Macro PR-AUC

In [10]:
class BasePairDataset(Dataset):
    def __init__(self, df):
        self.id1 = df.id1.values
        self.id2 = df.id2.values
        self.y = df.target.values.astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        a = self.id1[idx]
        b = self.id2[idx]
        return item_text[a], item_text[b], self.y[idx]

def collate_base(batch):
    t1, t2, y = zip(*batch)

    enc = tokenizer(
        list(t1),
        list(t2),
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )

    return enc, torch.tensor(y, dtype=torch.float32)

@torch.inference_mode()
def predict_base(model, df, batch_size=PRED_BATCH):
    model.eval()

    dl = DataLoader(
        BasePairDataset(df),
        batch_size=batch_size,
        collate_fn=collate_base,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    preds = []
    done = 0
    t0 = time.time()

    for batch_idx, (enc, _) in enumerate(dl, start=1):
        enc = {
            k: v.to(device, non_blocking=True)
            for k, v in enc.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            logits = model(**enc).logits.squeeze(-1)

        preds.append(torch.sigmoid(logits.float()).cpu().numpy())
        done += len(logits)

        if batch_idx % 500 == 0:
            speed = done / max(time.time()-t0, 1e-6)
            print(
                f"{done:,}/{len(df):,} ({speed:.0f} pair/s)",
                flush=True,
            )

    return np.concatenate(preds)

def macro_pr_auc(df, preds):
    z = df[["category", "target"]].reset_index(drop=True).copy()
    z["pred"] = np.asarray(preds)

    rows = []
    for cat, g in z.groupby("category", observed=True):
        y = g.target.to_numpy()
        p = g.pred.to_numpy()

        ap = np.nan if y.sum() == 0 else average_precision_score(y, p)

        rows.append({
            "category": cat,
            "pairs": len(g),
            "positive_rate": float(y.mean()),
            "PR_AUC": ap,
        })

    table = pd.DataFrame(rows).sort_values("PR_AUC").reset_index(drop=True)
    macro = float(table["PR_AUC"].dropna().mean())

    return macro, table

def make_balanced_fast_val(df, per_category=FAST_VAL_PER_CATEGORY, seed=0):
    parts = []
    for _, g in df.groupby("category", observed=True):
        n = min(per_category, len(g))
        parts.append(g.sample(n=n, random_state=seed))
    return pd.concat(parts, ignore_index=True)

## 9. Base V2 predictions: full LLM holdout

Эти predictions считаются один раз и затем переиспользуются на всех validation specialist.

In [11]:
t0 = time.time()

base_full_pred = predict_base(base_model, llm_val)
base_full_macro, base_full_table = macro_pr_auc(llm_val, base_full_pred)

print("\n" + "=" * 100)
print(f"BASE V2 FULL LLM MACRO: {base_full_macro:.9f}")
print(f"expected reference:     {EXPECTED_V2_FULL_MACRO:.9f}")
print(f"time: {(time.time()-t0)/60:.2f} min")
print("=" * 100)

display(base_full_table)

assert abs(base_full_macro - EXPECTED_V2_FULL_MACRO) < 0.004, (
    "V2 full metric воспроизвелась слишком далеко от reference. "
    "Остановись и проверь preprocessing/split."
)

32,000/191,555 (551 pair/s)
64,000/191,555 (549 pair/s)
96,000/191,555 (545 pair/s)
128,000/191,555 (541 pair/s)
160,000/191,555 (542 pair/s)

BASE V2 FULL LLM MACRO: 0.786482334
expected reference:     0.786482334
time: 5.89 min


,category,pairs,positive_rate,PR_AUC
0,Обувь,6199,0.072270,0.323267
1,Одежда,10744,0.054263,0.431439
2,Галантерея и аксессуары,11660,0.078302,0.541161
3,Ювелирные изделия,1265,0.181028,0.641862
4,Красота и гигиена,16959,0.201545,0.787953
5,Дом и сад,13003,0.214181,0.794009
6,Электроника,14657,0.088490,0.813678
7,Детские товары,11415,0.304424,0.816749
8,Канцелярские товары,7406,0.276533,0.818162
9,Спорт и отдых,9752,0.203651,0.822123


In [12]:
fast_val = make_balanced_fast_val(
    llm_val,
    FAST_VAL_PER_CATEGORY,
    FAST_VAL_SAMPLE_SEED,
)

fast_positions = fast_val["_val_pos"].values
base_fast_pred = base_full_pred[fast_positions]

base_fast_macro, base_fast_table = macro_pr_auc(
    fast_val,
    base_fast_pred,
)

print(f"BASE V2 FAST MACRO: {base_fast_macro:.9f}")
print(f"expected:           {EXPECTED_V2_FAST_MACRO:.9f}")
print(f"delta:              {base_fast_macro-EXPECTED_V2_FAST_MACRO:+.9f}")

assert abs(base_fast_macro - EXPECTED_V2_FAST_MACRO) < 0.004

BASE V2 FAST MACRO: 0.786378890
expected:           0.786378890
delta:              +0.000000000


## 10. Hard-example mining на balanced fashion pool

Hardness считаем как **soft-label BCE текущего V2**, умноженную на confidence LLM label.

То есть пары около `target=0.5` не будут доминировать mining только из-за неопределённости разметки.

In [13]:
t0 = time.time()

candidate_pred = predict_base(base_model, candidate_pool)

eps = 1e-6
p = np.clip(candidate_pred, eps, 1.0 - eps)
y = candidate_pool["target"].to_numpy(dtype=np.float32)

soft_bce = -(y * np.log(p) + (1.0-y) * np.log(1.0-p))
confidence = np.abs(2.0*y - 1.0)
confidence_w = CONFIDENCE_FLOOR + (
    1.0-CONFIDENCE_FLOOR
) * np.power(confidence, CONFIDENCE_POWER)

candidate_pool["_base_pred"] = candidate_pred
candidate_pool["_hardness"] = soft_bce * confidence_w

print(f"mining inference time: {(time.time()-t0)/60:.2f} min")
display(
    candidate_pool.groupby("category", observed=True)["_hardness"]
    .agg(["count", "mean", "median", "max"])
    .reset_index()
)

# Base predictions are fully cached now. Free GPU memory before specialist training.
base_model = base_model.to("cpu")
gc.collect()
torch.cuda.empty_cache()
print("Base V2 moved to CPU; GPU memory freed for specialist training.")


32,000/800,000 (596 pair/s)
64,000/800,000 (599 pair/s)
96,000/800,000 (599 pair/s)
128,000/800,000 (599 pair/s)
160,000/800,000 (599 pair/s)
192,000/800,000 (598 pair/s)
224,000/800,000 (595 pair/s)
256,000/800,000 (592 pair/s)
288,000/800,000 (589 pair/s)
320,000/800,000 (587 pair/s)
352,000/800,000 (585 pair/s)
384,000/800,000 (584 pair/s)
416,000/800,000 (582 pair/s)
448,000/800,000 (580 pair/s)
480,000/800,000 (578 pair/s)
512,000/800,000 (576 pair/s)
544,000/800,000 (575 pair/s)
576,000/800,000 (573 pair/s)
608,000/800,000 (571 pair/s)
640,000/800,000 (564 pair/s)
672,000/800,000 (558 pair/s)
704,000/800,000 (553 pair/s)
736,000/800,000 (548 pair/s)
768,000/800,000 (543 pair/s)
800,000/800,000 (539 pair/s)
mining inference time: 24.74 min


,category,count,mean,median,max
0,Галантерея и аксессуары,200000,0.247636,0.041149,6.778482
1,Обувь,200000,0.179550,0.019160,6.848718
2,Одежда,200000,0.154733,0.014610,6.833109
3,Ювелирные изделия,200000,0.279931,0.052684,6.817501


Base V2 moved to CPU; GPU memory freed for specialist training.


## 11. Формируем train V3

На категорию берём максимум `150k` пар:
- 50% самых hard;
- 30% random из оставшейся середины;
- 20% самых stable/easy.

Дополнительно hard rows имеют чуть больший sample weight.

In [14]:
def select_mined_train(g, n_total, seed):
    g = g.sort_values("_hardness", ascending=False).copy()
    n_total = min(n_total, len(g))

    n_hard = int(round(n_total * HARD_FRAC))
    n_random = int(round(n_total * RANDOM_FRAC))
    n_stable = n_total - n_hard - n_random

    hard = g.head(n_hard).copy()
    hard["_source"] = "hard"

    # Stable examples from the easiest end.
    remaining_after_hard = g.iloc[n_hard:].copy()
    stable = remaining_after_hard.tail(
        min(n_stable, len(remaining_after_hard))
    ).copy()
    stable["_source"] = "stable"

    used_idx = set(hard.index) | set(stable.index)
    middle = g.loc[~g.index.isin(used_idx)]

    n_random_eff = min(n_random, len(middle))
    random_part = middle.sample(
        n=n_random_eff,
        random_state=seed,
    ).copy()
    random_part["_source"] = "random"

    out = pd.concat(
        [hard, random_part, stable],
        ignore_index=True,
    )

    # If a very small category did not reach n_total due to rounding,
    # fill from unused rows.
    if len(out) < n_total:
        chosen_pos = set(out["_candidate_pos"].tolist())
        fill = g[
            ~g["_candidate_pos"].isin(chosen_pos)
        ].head(n_total-len(out)).copy()
        fill["_source"] = "random"
        out = pd.concat([out, fill], ignore_index=True)

    return out

parts = []
for cat, g in candidate_pool.groupby("category", observed=True):
    part = select_mined_train(
        g,
        TRAIN_PER_CATEGORY,
        SEED,
    )
    parts.append(part)

train_v3 = pd.concat(parts, ignore_index=True)
train_v3["_sample_weight"] = train_v3["_source"].map(SOURCE_WEIGHTS).astype(np.float32)

print("V3 train rows:", len(train_v3))
display(
    train_v3.groupby(
        ["category", "_source"],
        observed=True,
    ).size().rename("pairs").reset_index()
)

print("\nTarget stats:")
display(
    train_v3.groupby("category", observed=True)["target"]
    .agg(["count", "mean", "std"])
    .reset_index()
)

# Free mining-only dataframe.
del candidate_pool, candidate_pred, soft_bce, confidence, confidence_w, p, y
gc.collect()

V3 train rows: 600000


,category,_source,pairs
0,Галантерея и аксессуары,hard,75000
1,Галантерея и аксессуары,random,45000
2,Галантерея и аксессуары,stable,30000
3,Обувь,hard,75000
4,Обувь,random,45000
5,Обувь,stable,30000
6,Одежда,hard,75000
7,Одежда,random,45000
8,Одежда,stable,30000
9,Ювелирные изделия,hard,75000



Target stats:


,category,count,mean,std
0,Галантерея и аксессуары,150000,0.144076,0.298035
1,Обувь,150000,0.087900,0.245747
2,Одежда,150000,0.080167,0.228092
3,Ювелирные изделия,150000,0.165261,0.305694


0

## 12. Variant-aware specialist Dataset

Pair signal строится **после random swap**, поэтому значения `1/2` остаются согласованы с двумя transformer sequences.

In [15]:
class SpecialistDataset(Dataset):
    def __init__(self, df, training=False):
        self.id1 = df.id1.values
        self.id2 = df.id2.values
        self.y = df.target.values.astype(np.float32)
        self.training = training

        if "_sample_weight" in df.columns:
            self.sample_w = df["_sample_weight"].values.astype(np.float32)
        else:
            self.sample_w = np.ones(len(df), dtype=np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        a = self.id1[idx]
        b = self.id2[idx]

        if self.training and random.random() < RANDOM_SWAP_PROB:
            a, b = b, a

        signal = make_pair_signal(a, b)

        # Same short relational summary is visible from both sequences.
        t1 = item_text[a] + signal
        t2 = item_text[b] + signal

        return t1, t2, self.y[idx], self.sample_w[idx]

def collate_specialist(batch):
    t1, t2, y, sw = zip(*batch)

    enc = tokenizer(
        list(t1),
        list(t2),
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )

    return (
        enc,
        torch.tensor(y, dtype=torch.float32),
        torch.tensor(sw, dtype=torch.float32),
    )

def confidence_weight_torch(y):
    confidence = torch.abs(2.0*y - 1.0)
    return CONFIDENCE_FLOOR + (
        1.0-CONFIDENCE_FLOOR
    ) * confidence.pow(CONFIDENCE_POWER)

def weighted_soft_bce(logits, y, sample_w):
    per_example = F.binary_cross_entropy_with_logits(
        logits,
        y,
        reduction="none",
    )

    w = sample_w * confidence_weight_torch(y)

    return (per_example*w).sum() / w.sum().clamp_min(1e-6)

@torch.inference_mode()
def predict_specialist(model, df, batch_size=PRED_BATCH):
    model.eval()

    dl = DataLoader(
        SpecialistDataset(df, training=False),
        batch_size=batch_size,
        collate_fn=collate_specialist,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    preds = []
    for enc, _, _ in dl:
        enc = {
            k: v.to(device, non_blocking=True)
            for k, v in enc.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            logits = model(**enc).logits.squeeze(-1)

        preds.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )

    return np.concatenate(preds)

## 13. Specialist starts exactly from V2 weights

In [16]:
specialist = AutoModelForSequenceClassification.from_config(config)
specialist.load_state_dict(v2_ckpt["model"], strict=True)
specialist = specialist.to(device)

head_names = ("classifier", "score", "classification_head")

backbone_params = []
head_params = []

for name, param in specialist.named_parameters():
    if not param.requires_grad:
        continue
    if any(h in name.lower() for h in head_names):
        head_params.append(param)
    else:
        backbone_params.append(param)

print("backbone tensors:", len(backbone_params))
print("head tensors:", len(head_params))
assert head_params

backbone tensors: 199
head tensors: 2


## 14. Hybrid evaluator

Для 16 non-fashion категорий predictions **всегда остаются V2**.

Для каждой fashion-категории specialist включается только если он превосходит base минимум на `GATE_MARGIN`.

In [17]:
fast_fashion_mask = fast_val["category"].isin(FASHION_CATEGORIES).to_numpy()
fast_fashion_df = fast_val.loc[fast_fashion_mask].reset_index(drop=True)

base_fast_by_cat = {
    row["category"]: float(row["PR_AUC"])
    for _, row in base_fast_table.iterrows()
}

def evaluate_hybrid_fast(model, verbose=True):
    spec_pred_fashion = predict_specialist(
        model,
        fast_fashion_df,
    )

    spec_macro_f, spec_table_f = macro_pr_auc(
        fast_fashion_df,
        spec_pred_fashion,
    )

    spec_by_cat = {
        row["category"]: float(row["PR_AUC"])
        for _, row in spec_table_f.iterrows()
    }

    routing = {}
    for cat in sorted(FASHION_CATEGORIES):
        base_ap = base_fast_by_cat.get(cat, np.nan)
        spec_ap = spec_by_cat.get(cat, np.nan)

        use_spec = (
            np.isfinite(base_ap)
            and np.isfinite(spec_ap)
            and spec_ap > base_ap + GATE_MARGIN
        )

        routing[cat] = "specialist" if use_spec else "base"

    hybrid_pred = base_fast_pred.copy()

    # Fill only fashion rows routed to specialist.
    fashion_indices = np.where(fast_fashion_mask)[0]

    for local_idx, global_idx in enumerate(fashion_indices):
        cat = fast_val.iloc[global_idx]["category"]
        if routing.get(cat) == "specialist":
            hybrid_pred[global_idx] = spec_pred_fashion[local_idx]

    hybrid_macro, hybrid_table = macro_pr_auc(
        fast_val,
        hybrid_pred,
    )

    compare_rows = []
    for cat in sorted(FASHION_CATEGORIES):
        compare_rows.append({
            "category": cat,
            "base_AP": base_fast_by_cat.get(cat, np.nan),
            "specialist_AP": spec_by_cat.get(cat, np.nan),
            "route": routing.get(cat, "base"),
        })

    compare = pd.DataFrame(compare_rows)

    if verbose:
        print(f"specialist fashion macro: {spec_macro_f:.6f}")
        print(f"hybrid overall macro:     {hybrid_macro:.6f}")
        display(compare)

    return {
        "hybrid_macro": hybrid_macro,
        "routing": routing,
        "compare": compare,
        "hybrid_table": hybrid_table,
        "specialist_fashion_macro": spec_macro_f,
    }

baseline_eval = evaluate_hybrid_fast(specialist)

print("\nAt step 0 specialist == V2 weights, but input has new variant signals.")
print("This number is only a starting point, not a required sanity equality.")

specialist fashion macro: 0.452665
hybrid overall macro:     0.786379


,category,base_AP,specialist_AP,route
0,Галантерея и аксессуары,0.550660,0.507405,base
1,Обувь,0.311605,0.268513,base
2,Одежда,0.439770,0.397536,base
3,Ювелирные изделия,0.641862,0.637205,base



At step 0 specialist == V2 weights, but input has new variant signals.
This number is only a starting point, not a required sanity equality.


## 15. Optimizer / scheduler / full resume checkpoint

In [18]:
train_ds = SpecialistDataset(train_v3, training=True)

train_dl = DataLoader(
    train_ds,
    batch_size=MICRO_BATCH,
    collate_fn=collate_specialist,
    shuffle=True,
    drop_last=True,
    num_workers=0,
    pin_memory=True,
)

micro_steps_per_epoch = len(train_dl)
opt_steps_per_epoch = math.ceil(micro_steps_per_epoch / GRAD_ACCUM)
total_opt_steps = opt_steps_per_epoch * EPOCHS
warmup_steps = max(50, int(total_opt_steps * WARMUP_RATIO))

optimizer = torch.optim.AdamW(
    [
        {
            "params": backbone_params,
            "lr": BACKBONE_LR,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": head_params,
            "lr": HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
        },
    ]
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_opt_steps,
)

scaler = torch.amp.GradScaler("cuda")

print("train rows:", len(train_v3))
print("micro steps / epoch:", micro_steps_per_epoch)
print("optimizer steps / epoch:", opt_steps_per_epoch)
print("total optimizer steps:", total_opt_steps)
print("warmup steps:", warmup_steps)

def capture_rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all(),
    }

def save_resume(path, model, optimizer, scheduler, scaler, global_step, epoch, best_metric):
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "global_step": int(global_step),
            "epoch": int(epoch),
            "best_metric": float(best_metric),
            "rng": capture_rng_state(),
            "config": {
                "model_name": MODEL_NAME,
                "max_len": MAX_LEN,
                "max_attr_chars": MAX_ATTR_CHARS,
                "backbone_lr": BACKBONE_LR,
                "head_lr": HEAD_LR,
                "effective_batch": EFFECTIVE_BATCH,
            },
        },
        path,
    )

def save_best(path, model, eval_result, global_step):
    torch.save(
        {
            "model": model.state_dict(),
            "global_step": int(global_step),
            "hybrid_macro": float(eval_result["hybrid_macro"]),
            "routing": eval_result["routing"],
            "specialist_fashion_macro": float(
                eval_result["specialist_fashion_macro"]
            ),
            "compare": eval_result["compare"].to_dict(orient="records"),
            "model_name": MODEL_NAME,
            "max_len": MAX_LEN,
            "max_attr_chars": MAX_ATTR_CHARS,
        },
        path,
    )

train rows: 600000
micro steps / epoch: 18750
optimizer steps / epoch: 2344
total optimizer steps: 4688
warmup steps: 187


## 16. Training V3

Best checkpoint выбирается по **hybrid fast overall Macro PR-AUC**.

Resume checkpoint пишется независимо от best metric.

In [19]:
global_opt_step = 0

# Save a safe step-0 specialist checkpoint.
step0_eval = evaluate_hybrid_fast(specialist, verbose=False)
best_metric = float(step0_eval["hybrid_macro"])
best_routing = step0_eval["routing"].copy()
save_best(BEST_PATH, specialist, step0_eval, global_step=0)

print(
    f"step-0 hybrid macro={best_metric:.6f}; "
    f"base fast macro={base_fast_macro:.6f}"
)

t0 = time.time()
running_loss = 0.0
running_count = 0

optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    print(f"\n===== V3 EPOCH {epoch+1}/{EPOCHS} =====", flush=True)

    for micro_step, (enc, y, sample_w) in enumerate(train_dl, start=1):
        specialist.train()

        enc = {
            k: v.to(device, non_blocking=True)
            for k, v in enc.items()
        }
        y = y.to(device, non_blocking=True)
        sample_w = sample_w.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            logits = specialist(**enc).logits.squeeze(-1)
            loss = weighted_soft_bce(
                logits,
                y,
                sample_w,
            )
            loss_backward = loss / GRAD_ACCUM

        scaler.scale(loss_backward).backward()

        running_loss += float(loss.item())
        running_count += 1

        do_step = (
            micro_step % GRAD_ACCUM == 0
            or micro_step == len(train_dl)
        )

        if not do_step:
            continue

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            specialist.parameters(),
            MAX_GRAD_NORM,
        )

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        global_opt_step += 1

        if global_opt_step % 100 == 0:
            elapsed = time.time() - t0
            seen = (
                (epoch * len(train_dl) + micro_step)
                * MICRO_BATCH
            )
            speed = seen / max(elapsed, 1e-6)

            print(
                f"step {global_opt_step:5d}/{total_opt_steps} "
                f"loss={running_loss/max(running_count,1):.5f} "
                f"{speed:.0f} pair/s",
                flush=True,
            )

            running_loss = 0.0
            running_count = 0

        if (
            global_opt_step % RESUME_EVERY_OPT_STEPS == 0
            or global_opt_step == total_opt_steps
        ):
            save_resume(
                RESUME_PATH,
                specialist,
                optimizer,
                scheduler,
                scaler,
                global_opt_step,
                epoch,
                best_metric,
            )
            print("[RESUME] saved:", RESUME_PATH, flush=True)

        if (
            global_opt_step % EVAL_EVERY_OPT_STEPS == 0
            or global_opt_step == total_opt_steps
        ):
            eval_result = evaluate_hybrid_fast(
                specialist,
                verbose=True,
            )

            current_metric = eval_result["hybrid_macro"]

            print(
                f"[VAL] step={global_opt_step} "
                f"hybrid_macro={current_metric:.6f} "
                f"base={base_fast_macro:.6f}",
                flush=True,
            )

            if current_metric > best_metric:
                best_metric = current_metric
                best_routing = eval_result["routing"].copy()

                save_best(
                    BEST_PATH,
                    specialist,
                    eval_result,
                    global_step=global_opt_step,
                )

                print(
                    f"[BEST] saved metric={best_metric:.6f}",
                    flush=True,
                )

            gc.collect()
            torch.cuda.empty_cache()

print("\nTraining finished.")
print("best hybrid fast macro:", best_metric)
print("best routing:", best_routing)
print(f"total time: {(time.time()-t0)/3600:.2f} h")

step-0 hybrid macro=0.786379; base fast macro=0.786379

===== V3 EPOCH 1/2 =====
step   100/4688 loss=0.32146 197 pair/s
step   200/4688 loss=0.31882 197 pair/s
step   300/4688 loss=0.31420 197 pair/s
step   400/4688 loss=0.31186 198 pair/s
step   500/4688 loss=0.31186 198 pair/s
[RESUME] saved: /kaggle/working/e5_v3_resume.pt
specialist fashion macro: 0.490779
hybrid overall macro:     0.787984


,category,base_AP,specialist_AP,route
0,Галантерея и аксессуары,0.550660,0.537771,base
1,Обувь,0.311605,0.329966,specialist
2,Одежда,0.439770,0.442645,specialist
3,Ювелирные изделия,0.641862,0.652735,specialist


[VAL] step=500 hybrid_macro=0.787984 base=0.786379
[BEST] saved metric=0.787984
step   600/4688 loss=0.32092 190 pair/s
step   700/4688 loss=0.31736 191 pair/s
step   800/4688 loss=0.31720 192 pair/s
step   900/4688 loss=0.31336 193 pair/s
step  1000/4688 loss=0.31126 193 pair/s
[RESUME] saved: /kaggle/working/e5_v3_resume.pt
specialist fashion macro: 0.495586
hybrid overall macro:     0.788569


,category,base_AP,specialist_AP,route
0,Галантерея и аксессуары,0.550660,0.545308,base
1,Обувь,0.311605,0.324587,specialist
2,Одежда,0.439770,0.452456,specialist
3,Ювелирные изделия,0.641862,0.659994,specialist


[VAL] step=1000 hybrid_macro=0.788569 base=0.786379
[BEST] saved metric=0.788569
step  1100/4688 loss=0.31081 189 pair/s
step  1200/4688 loss=0.31406 190 pair/s
step  1300/4688 loss=0.31749 191 pair/s
step  1400/4688 loss=0.31095 191 pair/s
step  1500/4688 loss=0.32122 191 pair/s
[RESUME] saved: /kaggle/working/e5_v3_resume.pt
specialist fashion macro: 0.502820
hybrid overall macro:     0.789733


...

[VAL] step=4000 hybrid_macro=0.790609 base=0.786379
[BEST] saved metric=0.790609
step  4100/4688 loss=0.30698 189 pair/s
step  4200/4688 loss=0.30771 189 pair/s
step  4300/4688 loss=0.30843 189 pair/s
step  4400/4688 loss=0.30767 189 pair/s
step  4500/4688 loss=0.30880 189 pair/s
[RESUME] saved: /kaggle/working/e5_v3_resume.pt
specialist fashion macro: 0.505748
hybrid overall macro:     0.790358


,category,base_AP,specialist_AP,route
0,Галантерея и аксессуары,0.550660,0.550166,base
1,Обувь,0.311605,0.343934,specialist
2,Одежда,0.439770,0.466957,specialist
3,Ювелирные изделия,0.641862,0.661933,specialist


[VAL] step=4500 hybrid_macro=0.790358 base=0.786379
step  4600/4688 loss=0.30551 188 pair/s
[RESUME] saved: /kaggle/working/e5_v3_resume.pt
specialist fashion macro: 0.506459
hybrid overall macro:     0.790446


,category,base_AP,specialist_AP,route
0,Галантерея и аксессуары,0.550660,0.551254,base
1,Обувь,0.311605,0.344723,specialist
2,Одежда,0.439770,0.467752,specialist
3,Ювелирные изделия,0.641862,0.662106,specialist


[VAL] step=4688 hybrid_macro=0.790446 base=0.786379

Training finished.
best hybrid fast macro: 0.7906086444524086
best routing: {'Галантерея и аксессуары': 'base', 'Обувь': 'specialist', 'Одежда': 'specialist', 'Ювелирные изделия': 'specialist'}
total time: 1.78 h


## 17. Load best specialist + full hybrid validation

Routing берётся из **best fast validation checkpoint** и не переоптимизируется на full holdout.

In [20]:
best_ckpt = torch.load(
    BEST_PATH,
    map_location="cpu",
    weights_only=False,
)

specialist.load_state_dict(
    best_ckpt["model"],
    strict=True,
)
specialist = specialist.to(device)
specialist.eval()

routing = best_ckpt["routing"]

print("best step:", best_ckpt["global_step"])
print("best fast hybrid macro:", best_ckpt["hybrid_macro"])
print("routing:", routing)

best step: 4000
best fast hybrid macro: 0.7906086444524086
routing: {'Галантерея и аксессуары': 'base', 'Обувь': 'specialist', 'Одежда': 'specialist', 'Ювелирные изделия': 'specialist'}


In [21]:
full_fashion_mask = llm_val["category"].isin(FASHION_CATEGORIES).to_numpy()
full_fashion_df = llm_val.loc[full_fashion_mask].reset_index(drop=True)

t0 = time.time()
spec_full_fashion_pred = predict_specialist(
    specialist,
    full_fashion_df,
)

spec_full_fashion_macro, spec_full_fashion_table = macro_pr_auc(
    full_fashion_df,
    spec_full_fashion_pred,
)

hybrid_full_pred = base_full_pred.copy()
fashion_global_indices = np.where(full_fashion_mask)[0]

for local_idx, global_idx in enumerate(fashion_global_indices):
    cat = llm_val.iloc[global_idx]["category"]
    if routing.get(cat) == "specialist":
        hybrid_full_pred[global_idx] = spec_full_fashion_pred[local_idx]

hybrid_full_macro, hybrid_full_table = macro_pr_auc(
    llm_val,
    hybrid_full_pred,
)

print("\n" + "=" * 100)
print(f"BASE V2 FULL MACRO:        {base_full_macro:.9f}")
print(f"SPECIALIST FASHION MACRO: {spec_full_fashion_macro:.9f}")
print(f"HYBRID V3 FULL MACRO:      {hybrid_full_macro:.9f}")
print(f"DELTA vs V2:               {hybrid_full_macro-base_full_macro:+.9f}")
print(f"time: {(time.time()-t0)/60:.2f} min")
print("=" * 100)

base_full_by_cat = {
    row["category"]: float(row["PR_AUC"])
    for _, row in base_full_table.iterrows()
}
spec_full_by_cat = {
    row["category"]: float(row["PR_AUC"])
    for _, row in spec_full_fashion_table.iterrows()
}

comparison_rows = []
for cat in sorted(FASHION_CATEGORIES):
    comparison_rows.append({
        "category": cat,
        "base_AP": base_full_by_cat.get(cat, np.nan),
        "specialist_AP": spec_full_by_cat.get(cat, np.nan),
        "route_from_fast_val": routing.get(cat, "base"),
        "delta_specialist_vs_base": (
            spec_full_by_cat.get(cat, np.nan)
            - base_full_by_cat.get(cat, np.nan)
        ),
    })

full_compare = pd.DataFrame(comparison_rows)
display(full_compare)
display(hybrid_full_table)


BASE V2 FULL MACRO:        0.786482334
SPECIALIST FASHION MACRO: 0.504527989
HYBRID V3 FULL MACRO:      0.790182347
DELTA vs V2:               +0.003700013
time: 1.23 min


,category,base_AP,specialist_AP,route_from_fast_val,delta_specialist_vs_base
0,Галантерея и аксессуары,0.541161,0.547544,base,0.006383
1,Обувь,0.323267,0.348282,specialist,0.025015
2,Одежда,0.431439,0.457674,specialist,0.026235
3,Ювелирные изделия,0.641862,0.664612,specialist,0.022751


,category,pairs,positive_rate,PR_AUC
0,Обувь,6199,0.072270,0.348282
1,Одежда,10744,0.054263,0.457674
2,Галантерея и аксессуары,11660,0.078302,0.541161
3,Ювелирные изделия,1265,0.181028,0.664612
4,Красота и гигиена,16959,0.201545,0.787953
5,Дом и сад,13003,0.214181,0.794009
6,Электроника,14657,0.088490,0.813678
7,Детские товары,11415,0.304424,0.816749
8,Канцелярские товары,7406,0.276533,0.818162
9,Спорт и отдых,9752,0.203651,0.822123


## 18. Rough leaderboard signal

Это **не гарантия**. Просто сравниваем новую local metric с фактической парой V2:

`0.78648 local → 0.48388 LB`.

Для модели с category routing зависимость может измениться.

In [22]:
empirical_ratio = V2_PUBLIC_LB / base_full_macro
rough_lb = hybrid_full_macro * empirical_ratio

print("V2 empirical LB/local ratio:", empirical_ratio)
print("rough V3 LB signal:", rough_lb)
print("current V2 LB:", V2_PUBLIC_LB)
print("leader reference at design time: ~0.55")

V2 empirical LB/local ratio: 0.61524047419076
rough V3 LB signal: 0.48615216156698215
current V2 LB: 0.4838757641
leader reference at design time: ~0.55


## 19. Export: base + specialist + tokenizer + routing

Submission сможет маршрутизировать пары по category и прогонять каждую пару **ровно через один эксперт**, а не через обе модели.

In [23]:
# Put both models on CPU for export.
base_model = base_model.to("cpu")
specialist = specialist.to("cpu")

base_export = os.path.join(EXPORT_ROOT, "base_model")
spec_export = os.path.join(EXPORT_ROOT, "fashion_specialist")
tok_export = os.path.join(EXPORT_ROOT, "tokenizer")

os.makedirs(base_export, exist_ok=True)
os.makedirs(spec_export, exist_ok=True)
os.makedirs(tok_export, exist_ok=True)

base_model.save_pretrained(
    base_export,
    safe_serialization=True,
)
specialist.save_pretrained(
    spec_export,
    safe_serialization=True,
)
tokenizer.save_pretrained(tok_export)

routing_payload = {
    "fashion_categories": sorted(FASHION_CATEGORIES),
    "routing": routing,
    "max_len": MAX_LEN,
    "max_attr_chars": MAX_ATTR_CHARS,
    "variant_names": list(VARIANT_NAMES),
}

with open(
    os.path.join(EXPORT_ROOT, "routing.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(routing_payload, f, ensure_ascii=False, indent=2)

full_compare.to_csv(
    os.path.join(EXPORT_ROOT, "fashion_comparison.csv"),
    index=False,
)

hybrid_full_table.to_csv(
    os.path.join(EXPORT_ROOT, "hybrid_full_by_category.csv"),
    index=False,
)

metrics = {
    "experiment": "E5-small V3 fashion specialist + variant signals + hard mining",
    "base_model": MODEL_NAME,
    "base_checkpoint_step": EXPECTED_V2_STEP,
    "base_public_lb": V2_PUBLIC_LB,
    "base_full_llm_macro": float(base_full_macro),
    "best_specialist_step": int(best_ckpt["global_step"]),
    "best_fast_hybrid_macro": float(best_ckpt["hybrid_macro"]),
    "specialist_full_fashion_macro": float(spec_full_fashion_macro),
    "hybrid_full_llm_macro": float(hybrid_full_macro),
    "hybrid_delta_vs_v2": float(hybrid_full_macro-base_full_macro),
    "routing": routing,
    "max_len": MAX_LEN,
    "candidate_per_category": CANDIDATE_PER_CATEGORY,
    "train_per_category": TRAIN_PER_CATEGORY,
    "epochs": EPOCHS,
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "effective_batch": EFFECTIVE_BATCH,
    "hard_frac": HARD_FRAC,
    "random_frac": RANDOM_FRAC,
    "stable_frac": STABLE_FRAC,
}

with open(
    os.path.join(EXPORT_ROOT, "metrics.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

zip_path = shutil.make_archive(
    "/kaggle/working/e5_v3_hybrid_export",
    "zip",
    root_dir="/kaggle/working",
    base_dir="e5_v3_hybrid_export",
)

print("EXPORT:", EXPORT_ROOT)
print("ZIP:", zip_path)
print(f"ZIP size: {os.path.getsize(zip_path)/1024**3:.3f} GB")
print(json.dumps(metrics, ensure_ascii=False, indent=2))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

EXPORT: /kaggle/working/e5_v3_hybrid_export
ZIP: /kaggle/working/e5_v3_hybrid_export.zip
ZIP size: 0.669 GB
{
  "experiment": "E5-small V3 fashion specialist + variant signals + hard mining",
  "base_model": "intfloat/multilingual-e5-small",
  "base_checkpoint_step": 30000,
  "base_public_lb": 0.4838757641,
  "base_full_llm_macro": 0.7864823339791696,
  "best_specialist_step": 4000,
  "best_fast_hybrid_macro": 0.7906086444524086,
  "specialist_full_fashion_macro": 0.5045279889491541,
  "hybrid_full_llm_macro": 0.79018234651488,
  "hybrid_delta_vs_v2": 0.0037000125357103952,
  "routing": {
    "Галантерея и аксессуары": "base",
    "Обувь": "specialist",
    "Одежда": "specialist",
    "Ювелирные изделия": "specialist"
  },
  "max_len": 192,
  "candidate_per_category": 200000,
  "train_per_category": 150000,
  "epochs": 2,
  "backbone_lr": 8e-06,
  "head_lr": 3e-05,
  "effective_batch": 256,
  "hard_frac": 0.5,
  "random_frac": 0.3,
  "stable_frac": 0.2
}


## 20. Team / GitHub run summary

In [24]:
summary = f"""# E5 V3 — Fashion Specialist

## Reference V2

- Public LB: **{V2_PUBLIC_LB:.10f}**
- Backbone: `{MODEL_NAME}`
- checkpoint: step `{EXPECTED_V2_STEP}`
- full LLM group holdout Macro PR-AUC: `{base_full_macro:.6f}`

## V3

- Specialist initialized from V2 weights
- Fashion categories:
  - Обувь
  - Одежда
  - Галантерея и аксессуары
  - Ювелирные изделия
- Training distribution: LLM only
- Balanced candidate pool: up to `{CANDIDATE_PER_CATEGORY:,}` / category
- Mined train: up to `{TRAIN_PER_CATEGORY:,}` / category
- hard/random/stable = `{HARD_FRAC:.0%}/{RANDOM_FRAC:.0%}/{STABLE_FRAC:.0%}`
- Variant signals: size/color/article/model/gender/material
- epochs: `{EPOCHS}`
- LR: backbone `{BACKBONE_LR}`, head `{HEAD_LR}`
- MAX_LEN: `{MAX_LEN}`

## Result

- Best specialist step: `{best_ckpt["global_step"]}`
- Best fast hybrid Macro PR-AUC: `{best_ckpt["hybrid_macro"]:.6f}`
- Base full LLM Macro PR-AUC: `{base_full_macro:.6f}`
- Specialist fashion full Macro PR-AUC: `{spec_full_fashion_macro:.6f}`
- **Hybrid V3 full LLM Macro PR-AUC: `{hybrid_full_macro:.6f}`**
- Delta vs V2: `{hybrid_full_macro-base_full_macro:+.6f}`

## Routing

```json
{json.dumps(routing, ensure_ascii=False, indent=2)}
```

## Export

`/kaggle/working/e5_v3_hybrid_export.zip`
"""

summary_path = "/kaggle/working/e5_v3_RUN_SUMMARY.md"

with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary)

print(summary)
print("\nSaved:", summary_path)

# E5 V3 — Fashion Specialist

## Reference V2

- Public LB: **0.4838757641**
- Backbone: `intfloat/multilingual-e5-small`
- checkpoint: step `30000`
- full LLM group holdout Macro PR-AUC: `0.786482`

## V3

- Specialist initialized from V2 weights
- Fashion categories:
  - Обувь
  - Одежда
  - Галантерея и аксессуары
  - Ювелирные изделия
- Training distribution: LLM only
- Balanced candidate pool: up to `200,000` / category
- Mined train: up to `150,000` / category
- hard/random/stable = `50%/30%/20%`
- Variant signals: size/color/article/model/gender/material
- epochs: `2`
- LR: backbone `8e-06`, head `3e-05`
- MAX_LEN: `192`

## Result

- Best specialist step: `4000`
- Best fast hybrid Macro PR-AUC: `0.790609`
- Base full LLM Macro PR-AUC: `0.786482`
- Specialist fashion full Macro PR-AUC: `0.504528`
- **Hybrid V3 full LLM Macro PR-AUC: `0.790182`**
- Delta vs V2: `+0.003700`

## Routing

```json
{
  "Галантерея и аксессуары": "base",
  "Обувь": "specialist",
  "Одежда": "specialist

# После выполнения

Сохрани из Kaggle Output:

1. `e5_v3_hybrid_export.zip`
2. `e5_v3_RUN_SUMMARY.md`
3. `e5_v3_best_specialist.pt`
4. `e5_v3_resume.pt`

Перед submission:
- сначала сравни `HYBRID V3 FULL MACRO` с V2;
- затем собери inference, который маршрутизирует пары по `routing.json`;
- benchmark на тестовом размере;
- только после успешного speed/format test отправляй новый submit.

Если V3 улучшает только часть fashion-категорий — это нормально: routing оставит V2 на остальных.